# 05 · The Data Model — Dunders & Inheritance — **Depth**

> **Pull this notebook when:** you want your objects to behave like built-ins — containers you can
> `len`, index, iterate, compare. The transformer modules and memory stores all benefit.

Depth here is the **data model**: how dunders hook into language operators, the eq/hash contract, and
how inheritance resolves via the MRO. Predict first.

---

## 5.1 — Predict: `__repr__` vs `__str__` and the fallback

```python
class A:
    def __repr__(self): return "A-repr"

class B:
    def __str__(self): return "B-str"

a, b = A(), B()
print(str(a))     # ? A defines only __repr__
print(str(b))     # ? B defines only __str__
print(repr(b))    # ? B defines no __repr__
print([a])        # ? what does a list show
```

In [ ]:
class A:
    def __repr__(self): return "A-repr"
class B:
    def __str__(self): return "B-str"
a, b = A(), B()
print(str(a))
print(str(b))
print(repr(b))
print([a])

<details>
<summary>▶ Reveal</summary>

`A-repr`, `B-str`, a default `<...B object at 0x...>`, and `[A-repr]`.

`str()` falls back to `__repr__` when `__str__` is missing — so defining `__repr__` alone gives you
*both*. The reverse isn't true: `repr(b)` does **not** fall back to `__str__`, so `B` shows the ugly
default repr. And containers (`[a]`) always use `__repr__` on their elements, never `__str__`.

The rule: **if you define only one, define `__repr__`** — it's the fallback for everything and the one
containers use. `__str__` is an optional "prettier for humans" override on top.

</details>

## 5.2 — Predict: define `__eq__`, lose `__hash__`

```python
class P:
    def __init__(self, v): self.v = v
    def __eq__(self, other): return self.v == other.v

p = P(1)
print(P(1) == P(1))     # ?
s = {p}                 # ? put it in a set
```
Does the last line work?

In [ ]:
class P:
    def __init__(self, v): self.v = v
    def __eq__(self, other): return self.v == other.v

p = P(1)
print(P(1) == P(1))
s = {p}

<details>
<summary>▶ Reveal</summary>

`True`, then the set line raises **`TypeError: unhashable type: 'P'`**.

Here's the mechanism: when you define `__eq__` *without* `__hash__`, Python automatically sets
`__hash__ = None`, making instances **unhashable** — so they can't go in a set or be dict keys. Why?
Because the eq/hash contract demands that equal objects hash equal. If you redefined equality but kept
the default identity-based hash, two equal objects could hash *differently*, silently breaking sets and
dicts. Python refuses to let that happen by disabling hashing until you define it deliberately.

The fix is the contract from Milestone 01: define `__hash__` to agree with `__eq__` (hash the same
fields you compare). Define `__eq__` and `__hash__` **together**, always — that's the lesson.

</details>

## 5.3 — Predict: `in` and iteration without `__contains__`

A class with only `__getitem__` and `__len__` — no `__iter__`, no `__contains__`. Predict whether
`in` and a `for` loop still work.

```python
class Seq:
    def __init__(self, data): self.data = data
    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

s = Seq([10, 20, 30])
print(20 in s)            # ? no __contains__ defined
print([x for x in s])     # ? no __iter__ defined
```

In [ ]:
class Seq:
    def __init__(self, data): self.data = data
    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

s = Seq([10, 20, 30])
print(20 in s)
print([x for x in s])

<details>
<summary>▶ Reveal</summary>

`True`, then `[10, 20, 30]` — both work, from `__getitem__` alone.

Python's data model has **fallbacks**. If there's no `__iter__`, iteration falls back to calling
`__getitem__` with `0, 1, 2, ...` until `IndexError` — which is exactly what a list does internally,
so `for x in s` works. And `in`, with no `__contains__`, falls back to iterating and comparing. So
implementing just `__getitem__` (plus `__len__`) gives you indexing, iteration, *and* membership for
free. Defining `__iter__`/`__contains__` explicitly is about *control and efficiency*, not basic
function. This is the "behave like a built-in" payoff of the data model.

</details>

## 5.4 — Predict: the diamond, and what `super()` follows

```python
class A:
    def who(self): return "A"
class B(A):
    def who(self): return "B->" + super().who()
class C(A):
    def who(self): return "C->" + super().who()
class D(B, C):
    def who(self): return "D->" + super().who()

print(D().who())              # ?
print([c.__name__ for c in D.__mro__])   # ? the resolution order
```

In [ ]:
class A:
    def who(self): return "A"
class B(A):
    def who(self): return "B->" + super().who()
class C(A):
    def who(self): return "C->" + super().who()
class D(B, C):
    def who(self): return "D->" + super().who()

print(D().who())
print([c.__name__ for c in D.__mro__])

<details>
<summary>▶ Reveal</summary>

`D->B->C->A`, and the MRO is `['D', 'B', 'C', 'A', 'object']`.

The surprise people expect is `D->B->A` (B's super is A). But `super()` does **not** mean "my parent" —
it means "the **next class in the MRO**." The MRO (method resolution order, computed by the C3
algorithm) linearizes the diamond as D, B, C, A. So inside `B.who`, `super()` points to `C`, not `A`,
because C is next after B in *D's* MRO. Each `super().who()` walks one step along that single ordering,
so every class runs exactly once: `D->B->C->A`.

The takeaway: in multiple inheritance, `super()` cooperates along the MRO, not a fixed parent link.
This is why `super()` (not `ParentClass.method(self)`) is the correct way to chain — it respects the
MRO so diamonds don't double-call or skip classes.

</details>

## 5.5 — Refactor: is-a misused into has-a

`Stack` below "is-a" `list` by inheritance — but that leaks every list method (`insert`, `sort`,
indexing), breaking the stack abstraction. Refactor `Stack` to **contain** a list (composition),
exposing only `push`, `pop`, and `__len__`. The test checks the stack works *and* that list methods
are NOT exposed.

In [ ]:
class Stack:
    # was: class Stack(list)  — inheritance leaks the whole list API
    def __init__(self):
        # YOUR CODE HERE — hold a private list
        pass
    def push(self, x):
        # YOUR CODE HERE
        pass
    def pop(self):
        # YOUR CODE HERE
        pass
    def __len__(self):
        # YOUR CODE HERE
        pass

# Tests
s = Stack()
s.push(1); s.push(2); s.push(3)
assert len(s) == 3
assert s.pop() == 3
assert s.pop() == 2
assert len(s) == 1
# the list API is NOT leaked — composition hides it
assert not hasattr(s, "insert")
assert not hasattr(s, "sort")
print("5.5 passed")

<details>
<summary>▶ Why composition wins here</summary>

`class Stack(list)` says a stack *is a* list — so it inherits `insert`, `sort`, `[]` indexing, slicing,
everything. That breaks the abstraction: a stack should only allow push/pop, but now a caller can
`s.insert(0, x)` and violate LIFO. Composition (`self._items = []`) makes the stack *have a* list and
expose only the operations that make sense. The rule of thumb: inherit when the subclass genuinely *is
a* specialized kind of the parent and should support its whole interface; compose when you just want to
*use* another object's capabilities behind your own, narrower interface. Most "reuse" situations are
composition. This matters when you build Mara's modules — they *have* components, they aren't subclasses
of them.

</details>

## ★ Milestone 05 — A custom container that behaves like a built-in

Synthesize the whole notebook. Build `Deck` (a deck of cards) **by composition** (it holds a list),
implementing the data-model dunders so it behaves like a real sequence:

- `__len__` → `len(deck)`
- `__getitem__` → `deck[i]` (and, for free, iteration and `in`)
- `__eq__` + `__hash__`... actually a deck is mutable, so it should be **equal by contents** but you'll
  see why hashing is omitted — instead implement `__eq__` only and confirm it's unhashable (the 5.2
  lesson, deliberately).
- `__repr__` → `"Deck([...])"`

Then build `SortedDeck(Deck)` that adds a `sort()` method and uses `super().__init__(...)`.

(build it below)

In [ ]:
class Deck:
    def __init__(self, cards):
        # YOUR CODE HERE — store the cards list
        pass
    def __len__(self):
        # YOUR CODE HERE
        pass
    def __getitem__(self, i):
        # YOUR CODE HERE
        pass
    def __eq__(self, other):
        # YOUR CODE HERE — equal by contents
        pass
    def __repr__(self):
        # YOUR CODE HERE — "Deck([...])"
        pass

class SortedDeck(Deck):
    def __init__(self, cards):
        # YOUR CODE HERE — call super().__init__
        pass
    def sort(self):
        # YOUR CODE HERE — sort the underlying cards in place
        pass

# Tests
d = Deck([3, 1, 2])
assert len(d) == 3                       # __len__
assert d[0] == 3                         # __getitem__
assert list(d) == [3, 1, 2]              # iteration via __getitem__ fallback
assert 2 in d                            # membership via the same fallback
assert d == Deck([3, 1, 2])             # equal by contents
assert d != Deck([9, 9])
assert repr(d) == "Deck([3, 1, 2])"      # __repr__

# defining __eq__ without __hash__ makes it unhashable (the 5.2 lesson)
unhashable = False
try:
    {d}
except TypeError:
    unhashable = True
assert unhashable

# subclass extends via super()
sd = SortedDeck([3, 1, 2])
assert isinstance(sd, Deck)              # is-a relationship holds here
sd.sort()
assert list(sd) == [1, 2, 3]
assert len(sd) == 3                      # inherited behavior still works
print("Milestone 05 passed")

<details>
<summary>▶ What you built</summary>

A sequence-like container by **composition** (`self.cards = list(cards)`) that gets `len`, indexing,
iteration, and `in` from just `__len__` + `__getitem__` (the 5.3 fallback). `__eq__` compares contents;
because the deck is *mutable*, you deliberately omit `__hash__`, and Python makes it unhashable — which
is correct (a mutable thing shouldn't be a dict key, exactly the eq/hash reasoning from 5.2). `Deck` is
a legitimate place for inheritance: `SortedDeck` genuinely *is a* Deck that adds sorting, and
`super().__init__` chains construction along the MRO. Composition for *holding* the data, inheritance
for *specializing* the type — both used where each fits. This is the pattern for any custom container
in the project.

</details>